# ML Foundations for Generative AI

Hands-on PyTorch implementations covering tensor operations, autograd, neural networks from scratch,
and key building blocks used in modern generative models.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from einops import rearrange

plt.style.use("seaborn-v0_8-whitegrid")
torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

---
## 1. Tensor Operations

Tensors are the fundamental data structure in PyTorch. They generalize scalars, vectors,
and matrices to arbitrary dimensions. Every operation in a neural network -- from storing
weights to computing activations -- is a tensor operation.

### Creating and inspecting tensors

We start by creating tensors of different ranks and inspecting their shapes.
In deep learning, the shape of a tensor carries semantic meaning:
batch size, sequence length, embedding dimension, etc.

In [ ]:
scalar = torch.tensor(3.14)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
rank3 = torch.randn(2, 3, 4)  # e.g. (batch, seq_len, features)

for name, t in [("scalar", scalar), ("vector", vector), ("matrix", matrix), ("rank-3", rank3)]:
    print(f"{name:8s} | shape: {str(t.shape):12s} | ndim: {t.ndim} | dtype: {t.dtype}")

### Reshaping and broadcasting

Reshaping rearranges elements without copying data. Broadcasting lets tensors of
different shapes participate in element-wise operations by virtually expanding
dimensions of size 1.

In [ ]:
# Reshape: flatten and unflatten
x = torch.arange(24).float()
print("Original:", x.shape)

x_reshaped = x.view(2, 3, 4)
print("Reshaped to (2,3,4):", x_reshaped.shape)

x_flat = x_reshaped.flatten(start_dim=1)  # keep batch dim, flatten rest
print("Flattened from dim 1:", x_flat.shape)

# Broadcasting: add a bias vector to every row of a matrix
data = torch.randn(4, 3)      # 4 samples, 3 features
bias = torch.tensor([10.0, 20.0, 30.0])  # shape (3,)
result = data + bias           # bias broadcasts to (4, 3)
print(f"\ndata shape: {data.shape}, bias shape: {bias.shape}, result shape: {result.shape}")
print("First row of data: ", data[0])
print("First row of result:", result[0])

### Batched matrix multiplication and einops

`torch.bmm` performs matrix multiplication across a batch dimension.
This is the core operation behind attention: Q @ K^T produces a
(batch, seq, seq) attention score matrix.

`einops.rearrange` provides a readable, declarative syntax for
reshaping tensors -- especially useful when manipulating multi-head
attention tensors.

In [ ]:
batch_size, seq_len, d_model = 2, 5, 8

queries = torch.randn(batch_size, seq_len, d_model)
keys = torch.randn(batch_size, seq_len, d_model)

# Batched matmul: (B, S, D) @ (B, D, S) -> (B, S, S)
attention_scores = torch.bmm(queries, keys.transpose(1, 2))
print("Attention scores shape:", attention_scores.shape)

# einops: simulate splitting into multi-head format
num_heads = 2
head_dim = d_model // num_heads

# (batch, seq, d_model) -> (batch, heads, seq, head_dim)
multi_head = rearrange(queries, "b s (h d) -> b h s d", h=num_heads)
print(f"Multi-head shape: {multi_head.shape}  (batch, heads, seq, head_dim)")

# Merge heads back
merged = rearrange(multi_head, "b h s d -> b s (h d)")
print(f"Merged back:      {merged.shape}")
print(f"Reconstruction matches original: {torch.allclose(queries, merged)}")

---
## 2. Autograd Deep Dive

PyTorch's autograd engine automatically computes gradients by recording operations
on tensors that have `requires_grad=True`. This builds a dynamic computational graph
that is traversed backward to compute derivatives via the chain rule.

### Manual vs autograd gradients

Consider f(x) = x^3 + 2x^2 - 5x.

The analytical derivative is f'(x) = 3x^2 + 4x - 5.

We compute this by hand and then verify with autograd.

In [ ]:
def f(x):
    return x**3 + 2 * x**2 - 5 * x

def f_prime_analytical(x):
    """f'(x) = 3x^2 + 4x - 5"""
    return 3 * x**2 + 4 * x - 5

# Evaluate at several points
test_points = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0, 3.0])

print(f"{'x':>6s} {'f(x)':>10s} {'manual f\'(x)':>14s} {'autograd f\'(x)':>16s} {'match':>8s}")
print("-" * 58)

for val in test_points:
    x = torch.tensor(val, requires_grad=True)
    y = f(x)
    y.backward()

    manual_grad = f_prime_analytical(val).item()
    auto_grad = x.grad.item()
    match = abs(manual_grad - auto_grad) < 1e-6

    print(f"{val.item():6.1f} {y.item():10.2f} {manual_grad:14.2f} {auto_grad:16.2f} {str(match):>8s}")

### Visualizing the computational graph

When PyTorch computes f(x) = x^3 + 2x^2 - 5x, it internally builds a directed
acyclic graph (DAG) of operations. Each intermediate result remembers which operation
created it (`grad_fn`). Backpropagation walks this graph in reverse.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

# Break f(x) into named intermediates to inspect the graph
a = x ** 3
b = 2 * x ** 2
c = 5 * x
y = a + b - c

print("Computational graph (grad_fn chain):")
print(f"  y        = {y.item():.1f},  grad_fn = {y.grad_fn}")
print(f"  a (x^3)  = {a.item():.1f},  grad_fn = {a.grad_fn}")
print(f"  b (2x^2) = {b.item():.1f},  grad_fn = {b.grad_fn}")
print(f"  c (5x)   = {c.item():.1f}, grad_fn = {c.grad_fn}")
print(f"  x        = {x.item():.1f},  is leaf = {x.is_leaf}")

y.backward()
print(f"\ndy/dx at x=3: {x.grad.item():.1f}  (expected: 3*9 + 4*3 - 5 = {3*9 + 4*3 - 5})")

### Gradient accumulation and zeroing

PyTorch accumulates gradients by default -- calling `.backward()` multiple times
adds to `.grad` instead of replacing it. This is useful for gradient accumulation
across micro-batches, but you must zero gradients between optimization steps.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

# First backward pass
y1 = x ** 2
y1.backward()
print(f"After first backward (y=x^2, dy/dx=2x=4):   x.grad = {x.grad.item():.1f}")

# Second backward pass WITHOUT zeroing -- gradients accumulate
y2 = x ** 2
y2.backward()
print(f"After second backward (accumulated):          x.grad = {x.grad.item():.1f}")

# Zero the gradient
x.grad.zero_()
print(f"After zeroing:                                x.grad = {x.grad.item():.1f}")

# Third backward pass -- clean gradient
y3 = x ** 2
y3.backward()
print(f"After third backward (fresh):                 x.grad = {x.grad.item():.1f}")

print("\n-- Practical use: simulating gradient accumulation over 4 micro-batches --")
w = torch.tensor(1.0, requires_grad=True)
micro_batches = [torch.tensor(v) for v in [1.0, 2.0, 3.0, 4.0]]

for i, mb in enumerate(micro_batches):
    loss = (w * mb - mb ** 2) ** 2  # dummy loss
    loss.backward()
    print(f"  micro-batch {i}: loss={loss.item():.2f}, accumulated grad={w.grad.item():.2f}")

print(f"  Total accumulated gradient: {w.grad.item():.2f}")

---
## 3. Building a Neural Network from Scratch

To understand what PyTorch's `nn.Module` does under the hood, we first build
a 2-layer MLP using raw tensor operations. The task: learn the sine function.
Given x in [-2pi, 2pi], predict sin(x).

### Prepare data

In [ ]:
torch.manual_seed(42)

num_samples = 500
x_train = torch.linspace(-2 * torch.pi, 2 * torch.pi, num_samples).unsqueeze(1)
y_train = torch.sin(x_train)

print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")

plt.figure(figsize=(8, 3))
plt.plot(x_train.numpy(), y_train.numpy(), label="sin(x)")
plt.xlabel("x")
plt.ylabel("sin(x)")
plt.title("Target function: sin(x)")
plt.legend()
plt.tight_layout()
plt.show()

### Manual MLP: forward pass, loss, backward, update

Architecture: input(1) -> hidden(64, ReLU) -> output(1)

We initialize weights with Kaiming uniform (good for ReLU), then run
a training loop doing everything by hand.

In [ ]:
torch.manual_seed(42)

hidden_size = 64
learning_rate = 0.01
num_epochs = 2000

# Initialize parameters (Kaiming uniform for ReLU)
w1 = torch.randn(1, hidden_size) * (2.0 / 1) ** 0.5
b1 = torch.zeros(1, hidden_size)
w2 = torch.randn(hidden_size, 1) * (2.0 / hidden_size) ** 0.5
b2 = torch.zeros(1, 1)

w1.requires_grad_(True)
b1.requires_grad_(True)
w2.requires_grad_(True)
b2.requires_grad_(True)

params = [w1, b1, w2, b2]
losses_manual = []

for epoch in range(num_epochs):
    # Forward pass
    hidden = x_train @ w1 + b1          # (500, 64)
    hidden_act = torch.relu(hidden)      # ReLU activation
    prediction = hidden_act @ w2 + b2    # (500, 1)

    # MSE loss
    loss = ((prediction - y_train) ** 2).mean()
    losses_manual.append(loss.item())

    # Backward pass
    loss.backward()

    # Parameter update (SGD)
    with torch.no_grad():
        for p in params:
            p -= learning_rate * p.grad
            p.grad.zero_()

    if epoch % 500 == 0 or epoch == num_epochs - 1:
        print(f"Epoch {epoch:5d} | Loss: {loss.item():.6f}")

# Final predictions
with torch.no_grad():
    preds_manual = torch.relu(x_train @ w1 + b1) @ w2 + b2

### Plot manual MLP results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses_manual)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Training Loss (Manual MLP)")
axes[0].set_yscale("log")

axes[1].plot(x_train.numpy(), y_train.numpy(), label="Ground truth", linewidth=2)
axes[1].plot(x_train.numpy(), preds_manual.numpy(), "--", label="Manual MLP", linewidth=2)
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("Predictions vs Ground Truth")
axes[1].legend()

plt.tight_layout()
plt.show()

total_params = sum(p.numel() for p in params)
print(f"Total parameters (manual): {total_params}")

---
## 4. The Same Network with nn.Module

Now we rewrite the exact same architecture using PyTorch's `nn.Module` and `nn.Linear`.
The optimizer (Adam) replaces our manual SGD update. The code is shorter, less error-prone,
and the same underlying math runs beneath it.

In [ ]:
class SineNet(nn.Module):
    def __init__(self, hidden_size=64):
        super().__init__()
        self.layer1 = nn.Linear(1, hidden_size)
        self.layer2 = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = self.layer2(x)
        return x

torch.manual_seed(42)
model = SineNet(hidden_size=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

losses_module = []

for epoch in range(2000):
    prediction = model(x_train)
    loss = criterion(prediction, y_train)
    losses_module.append(loss.item())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0 or epoch == 1999:
        print(f"Epoch {epoch:5d} | Loss: {loss.item():.6f}")

### Compare results and parameter counts

In [ ]:
with torch.no_grad():
    preds_module = model(x_train)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses_manual, label="Manual (SGD)", alpha=0.8)
axes[0].plot(losses_module, label="nn.Module (Adam)", alpha=0.8)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Training Loss Comparison")
axes[0].set_yscale("log")
axes[0].legend()

axes[1].plot(x_train.numpy(), y_train.numpy(), label="Ground truth", linewidth=2)
axes[1].plot(x_train.numpy(), preds_manual.numpy(), "--", label="Manual MLP", linewidth=2)
axes[1].plot(x_train.numpy(), preds_module.numpy(), "-.", label="nn.Module MLP", linewidth=2)
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("Predictions Comparison")
axes[1].legend()

plt.tight_layout()
plt.show()

module_params = sum(p.numel() for p in model.parameters())
print(f"\nParameter counts:")
print(f"  Manual MLP:     {total_params}")
print(f"  nn.Module MLP:  {module_params}")
print(f"  Match: {total_params == module_params}")

print(f"\nnn.Module architecture:")
for name, param in model.named_parameters():
    print(f"  {name:20s} shape={str(list(param.shape)):12s} numel={param.numel()}")

---
## 5. Key Operations from Scratch

Generative AI models rely on a handful of key operations.
Understanding their internals helps when debugging, optimizing,
or reading model code.

### Softmax from scratch

Softmax converts a vector of raw scores (logits) into a probability distribution.
The naive formula exp(x_i) / sum(exp(x_j)) overflows for large values.
The standard trick is to subtract the maximum value first.

In [ ]:
def softmax_naive(logits, dim=-1):
    """Naive softmax -- overflows for large logits."""
    exp_vals = torch.exp(logits)
    return exp_vals / exp_vals.sum(dim=dim, keepdim=True)

def softmax_stable(logits, dim=-1):
    """Numerically stable softmax: subtract max before exp."""
    shifted = logits - logits.max(dim=dim, keepdim=True).values
    exp_vals = torch.exp(shifted)
    return exp_vals / exp_vals.sum(dim=dim, keepdim=True)

# Normal-range logits: all three agree
logits_normal = torch.tensor([2.0, 1.0, 0.1])
print("Normal logits:")
print(f"  Naive:    {softmax_naive(logits_normal)}")
print(f"  Stable:   {softmax_stable(logits_normal)}")
print(f"  F.softmax:{F.softmax(logits_normal, dim=-1)}")

# Large logits: naive overflows
logits_large = torch.tensor([1000.0, 1001.0, 1002.0])
print("\nLarge logits (overflow scenario):")
print(f"  Naive:    {softmax_naive(logits_large)}")
print(f"  Stable:   {softmax_stable(logits_large)}")
print(f"  F.softmax:{F.softmax(logits_large, dim=-1)}")

# Verify on a batch
batch_logits = torch.randn(4, 10)
ours = softmax_stable(batch_logits, dim=-1)
ref = F.softmax(batch_logits, dim=-1)
print(f"\nBatch test -- max absolute error: {(ours - ref).abs().max().item():.2e}")
print(f"All probabilities sum to 1: {torch.allclose(ours.sum(dim=-1), torch.ones(4))}")

### Layer normalization from scratch

Layer normalization normalizes each sample independently across its features.
Unlike batch normalization, it does not depend on batch statistics, which makes
it the default choice for transformers and language models.

LayerNorm(x) = gamma * (x - mean) / sqrt(var + eps) + beta

In [ ]:
def layer_norm(x, gamma, beta, eps=1e-5):
    """Layer normalization over the last dimension."""
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    x_norm = (x - mean) / torch.sqrt(var + eps)
    return gamma * x_norm + beta

torch.manual_seed(42)
batch_size, features = 4, 8
x = torch.randn(batch_size, features)

# Our implementation
gamma = torch.ones(features)
beta = torch.zeros(features)
out_ours = layer_norm(x, gamma, beta)

# PyTorch reference
ln = nn.LayerNorm(features, elementwise_affine=True)
out_ref = ln(x)

print(f"Input mean per sample:  {x.mean(dim=-1).tolist()}")
print(f"Output mean per sample: {[f'{v:.6f}' for v in out_ours.mean(dim=-1).tolist()]}")
print(f"Output var per sample:  {[f'{v:.4f}' for v in out_ours.var(dim=-1, unbiased=False).tolist()]}")
print(f"\nMax absolute error vs nn.LayerNorm: {(out_ours - out_ref).abs().max().item():.2e}")

### Embedding lookup from scratch

An embedding layer is a learnable lookup table that maps integer token IDs
to dense vectors. It is equivalent to one-hot encoding followed by a linear
layer, but implemented as a direct index operation for efficiency.

In [ ]:
vocab_size = 10
embed_dim = 4

# Our embedding: just a weight matrix + integer indexing
torch.manual_seed(42)
embed_weight = torch.randn(vocab_size, embed_dim)

token_ids = torch.tensor([0, 3, 7, 3])  # a short sequence

# Lookup: index into the weight matrix
out_manual = embed_weight[token_ids]

# Equivalent via one-hot + matmul
one_hot = F.one_hot(token_ids, num_classes=vocab_size).float()
out_onehot = one_hot @ embed_weight

# PyTorch nn.Embedding with the same weights
emb = nn.Embedding(vocab_size, embed_dim)
with torch.no_grad():
    emb.weight.copy_(embed_weight)
out_pytorch = emb(token_ids)

print(f"Token IDs: {token_ids.tolist()}")
print(f"\nManual lookup (indexing):")
print(out_manual)
print(f"\nOne-hot @ weight:")
print(out_onehot)
print(f"\nnn.Embedding:")
print(out_pytorch)

print(f"\nAll three match: {torch.allclose(out_manual, out_onehot) and torch.allclose(out_manual, out_pytorch)}")
print(f"\nNote: tokens 3 appears twice -- both lookups return the same vector:")
print(f"  Position 1: {out_manual[1].tolist()}")
print(f"  Position 3: {out_manual[3].tolist()}")

---
## 6. Residual Connections

Residual (skip) connections are one of the most important architectural innovations
in deep learning. They allow gradients to flow through an identity shortcut, preventing
the vanishing gradient problem in deep networks. Every transformer block uses them.

### Define networks: with and without residual connections

In [ ]:
class PlainDeepNet(nn.Module):
    """A 10-layer network without residual connections."""
    def __init__(self, width=64):
        super().__init__()
        self.input_proj = nn.Linear(1, width)
        self.layers = nn.ModuleList([nn.Linear(width, width) for _ in range(10)])
        self.output_proj = nn.Linear(width, 1)

    def forward(self, x):
        x = torch.relu(self.input_proj(x))
        for layer in self.layers:
            x = torch.relu(layer(x))
        return self.output_proj(x)


class ResidualDeepNet(nn.Module):
    """A 10-layer network with residual connections."""
    def __init__(self, width=64):
        super().__init__()
        self.input_proj = nn.Linear(1, width)
        self.layers = nn.ModuleList([nn.Linear(width, width) for _ in range(10)])
        self.output_proj = nn.Linear(width, 1)

    def forward(self, x):
        x = torch.relu(self.input_proj(x))
        for layer in self.layers:
            x = torch.relu(layer(x)) + x  # <-- residual connection
        return self.output_proj(x)


print("PlainDeepNet:")
plain_net = PlainDeepNet()
print(f"  Parameters: {sum(p.numel() for p in plain_net.parameters())}")

print("ResidualDeepNet:")
res_net = ResidualDeepNet()
print(f"  Parameters: {sum(p.numel() for p in res_net.parameters())}")

### Compare gradient magnitudes across layers

We pass a single batch through each network, compute the loss, and then
inspect the gradient magnitude at each layer. In the plain network, gradients
tend to vanish (shrink toward zero) in early layers.

In [ ]:
torch.manual_seed(42)
plain_net = PlainDeepNet()
res_net = ResidualDeepNet()

x_sample = torch.randn(32, 1)
y_sample = torch.sin(x_sample)

def get_layer_gradient_norms(net, x, y):
    """Run a forward+backward pass and return gradient norms for each layer."""
    net.zero_grad()
    pred = net(x)
    loss = F.mse_loss(pred, y)
    loss.backward()

    grad_norms = []
    for i, layer in enumerate(net.layers):
        norm = layer.weight.grad.norm().item()
        grad_norms.append(norm)
    return grad_norms

plain_grads = get_layer_gradient_norms(plain_net, x_sample, y_sample)
res_grads = get_layer_gradient_norms(res_net, x_sample, y_sample)

print(f"{'Layer':>6s} {'Plain grad norm':>16s} {'Residual grad norm':>20s}")
print("-" * 44)
for i in range(10):
    print(f"{i:6d} {plain_grads[i]:16.6f} {res_grads[i]:20.6f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(np.arange(10) - 0.2, plain_grads, 0.4, label="Plain (no skip)", alpha=0.8)
ax.bar(np.arange(10) + 0.2, res_grads, 0.4, label="Residual (skip)", alpha=0.8)
ax.set_xlabel("Layer index")
ax.set_ylabel("Gradient norm")
ax.set_title("Gradient Magnitude by Layer")
ax.legend()
ax.set_xticks(range(10))
plt.tight_layout()
plt.show()

### Training comparison: loss curves

We train both networks on the sine task and compare convergence.
The residual network should train faster and reach a lower loss,
because gradients flow more effectively through skip connections.

In [ ]:
def train_network(net, x, y, num_epochs=3000, lr=0.001):
    """Train a network and return the loss history."""
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    losses = []
    for epoch in range(num_epochs):
        pred = net(x)
        loss = F.mse_loss(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if epoch % 1000 == 0 or epoch == num_epochs - 1:
            print(f"  Epoch {epoch:5d} | Loss: {loss.item():.6f}")
    return losses

torch.manual_seed(42)
plain_net_2 = PlainDeepNet()
print("Training PlainDeepNet (no residuals):")
plain_losses = train_network(plain_net_2, x_train, y_train)

torch.manual_seed(42)
res_net_2 = ResidualDeepNet()
print("\nTraining ResidualDeepNet (with residuals):")
res_losses = train_network(res_net_2, x_train, y_train)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(plain_losses, label="Plain (no residuals)", alpha=0.8)
axes[0].plot(res_losses, label="Residual (skip connections)", alpha=0.8)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Training Loss: Plain vs Residual")
axes[0].set_yscale("log")
axes[0].legend()

# Predictions
with torch.no_grad():
    preds_plain = plain_net_2(x_train)
    preds_res = res_net_2(x_train)

axes[1].plot(x_train.numpy(), y_train.numpy(), label="Ground truth", linewidth=2)
axes[1].plot(x_train.numpy(), preds_plain.numpy(), "--", label="Plain", linewidth=2, alpha=0.8)
axes[1].plot(x_train.numpy(), preds_res.numpy(), "-.", label="Residual", linewidth=2, alpha=0.8)
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("Final Predictions")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nFinal MSE -- Plain: {plain_losses[-1]:.6f}, Residual: {res_losses[-1]:.6f}")

---
## Summary

This notebook covered six foundational topics for generative AI:

1. **Tensor operations** -- creation, reshaping, broadcasting, batched matmul, and einops
2. **Autograd** -- analytical vs automatic differentiation, computational graphs, gradient accumulation
3. **Manual neural network** -- forward pass, loss, backward, SGD update with raw tensors
4. **nn.Module network** -- the same architecture expressed cleanly with PyTorch abstractions
5. **Key operations** -- softmax (with numerical stability), layer normalization, embeddings
6. **Residual connections** -- gradient flow comparison and training dynamics in deep networks

These building blocks appear in every generative model -- from GPT to diffusion models.
The next notebook will build on these to implement attention and transformer components.